In [1]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [2]:
rmprocs(workers())
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 4        # ← set as desired
addprocs(num_workers)

┌ Warning: rmprocs: process 1 not removed
└ @ Distributed C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\cluster.jl:1049


4-element Vector{Int64}:
 2
 3
 4
 5

In [3]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 300
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "RW"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
end

In [4]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [5]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

In [6]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [7]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 2:	[ Info: Performing boostrap simulation number 1
      From worker 5:	[ Info: Performing boostrap simulation number 4
      From worker 4:	[ Info: Performing boostrap simulation number 3
      From worker 3:	[ Info: Performing boostrap simulation number 2
      From worker 2:	[ Info: Bootstrap 1 generated.
      From worker 4:	[ Info: Bootstrap 3 generated.
      From worker 3:	[ Info: Bootstrap 2 generated.
      From worker 5:	[ Info: Bootstrap 4 generated.


Task (done) @0x000002972f79c3f0

In [8]:
sed_vals

4-element Vector{Vector{Float64}}:
 [NaN, 3.0455810942271212e-6, -0.00011449907975621704, -7.607900958976751e-5, -9.346785899262179e-5, -9.72911276874459e-5, -0.00010044462646340951, -0.00020053284388487573, -1.7744097867324662e-5, 6.9211574812292394e-6  …  -1.752451779260815e-5, -1.585152868019521e-5, -1.3317971262989301e-5, -1.0791162324245095e-5, -8.470190324736662e-6, -8.554870609367394e-6, -9.569188341559433e-6, -8.07409629480525e-6, -7.006524789376933e-6, -7.062539441766058e-6]
 [NaN, -4.433744812282084e-6, -0.0001891358308360478, 0.0004541847990363125, 0.0004993130896762458, 0.0002873844645471603, 9.782450851556115e-5, 4.7893409571705595e-5, 5.653387847508553e-7, 1.6405972836338277e-5  …  5.5706798381148395e-5, 6.270199673469594e-5, 5.538549282542459e-5, 5.476875827439278e-5, 5.14337411840643e-5, 6.976146243749585e-5, 0.00011612058527380072, 0.00010428111651926463, 9.276887364714108e-5, 7.95432644961103e-5]
 [NaN, -0.00011961192714948523, -8.286582058018267e-5, -0.00014338880956

In [9]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

4

In [10]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 0.0003880460437997215


In [11]:
# Save the SED values into BSON file
BSON.@save "sed_thresholds.bson" sed_vals thr